In [1]:
# ============================================================
# CELL 1 — PROJECT SETUP
# WHY:
# Install the required packages and verify that your
# previously uploaded datasets are still available.
# ============================================================

!pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    sentence-transformers \
    langchain \
    langchain-community \
    langchain-huggingface \
    langgraph \
    faiss-cpu \
    fastapi \
    uvicorn \
    python-multipart \
    pymupdf \
    pandas \
    numpy \
    scikit-learn \
    pydantic \
    pyngrok

print("✅ Packages installed!")

# ------------------------------------------------------------
# Verify existing files
# ------------------------------------------------------------

import os

print("\n📂 Existing project files:")

for file in os.listdir("/content"):
    print("📄", file)

print("\n🚀 CELL 1 COMPLETED!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
✅ Packages installed!

📂 Existing project files:
📄 .config
📄 resume_data.csv
📄 job_description.csv
📄 LinkedIn_RDB_three.csv
📄 Software Questions.csv
📄 .ipynb_checkpoints
📄 sample_d

In [2]:
# ============================================================
# CELL 2 — VERIFY ENVIRONMENT
# WHY:
# Confirm Python, PyTorch, Transformers, GPU and all
# important libraries before loading the AI models.
# ============================================================

import sys
import torch
import transformers
import sentence_transformers
import langchain
import langgraph
import fastapi
import pandas as pd
import fitz
import faiss

print("🐍 Python:", sys.version.split()[0])
print("🔥 PyTorch:", torch.__version__)
print("🤗 Transformers:", transformers.__version__)
print("🔤 Sentence Transformers:", sentence_transformers.__version__)
print("🦜 LangChain:", langchain.__version__)
print("🔗 LangGraph: imported successfully")
print("⚡ FastAPI:", fastapi.__version__)
print("🐼 Pandas:", pd.__version__)
print("📄 PyMuPDF: imported successfully")
print("🔎 FAISS: imported successfully")

print("\nGPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
    print(
        "GPU Memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("⚠️ GPU not available")

🐍 Python: 3.13.15
🔥 PyTorch: 2.11.0+cu130
🤗 Transformers: 5.17.0
🔤 Sentence Transformers: 5.7.0
🦜 LangChain: 1.4.0
🔗 LangGraph: imported successfully
⚡ FastAPI: 0.141.1
🐼 Pandas: 2.2.3
📄 PyMuPDF: imported successfully
🔎 FAISS: imported successfully

GPU Available: True
GPU: Tesla T4
CUDA: 13.0
GPU Memory: 14.56 GB


In [3]:
# ============================================================
# CELL 3 — LOAD BGE-LARGE
# WHY:
# BGE-large creates high-quality semantic embeddings for
# resumes and job descriptions.
# These embeddings will power our FAISS job retrieval.
# ============================================================

from sentence_transformers import SentenceTransformer
import torch

EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5"

print("⏳ Loading BGE-large...")

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cuda"
)

print("✅ BGE-large loaded successfully!")
print("📐 Embedding dimension:", embedding_model.get_sentence_embedding_dimension())
print("🖥️ Device:", embedding_model.device)

⏳ Loading BGE-large...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ BGE-large loaded successfully!
📐 Embedding dimension: 1024
🖥️ Device: cuda:0


/tmp/ipykernel_2222/4046067049.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("📐 Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


In [4]:
# ============================================================
# CELL 4 — LOAD JOB DATASET
# WHY:
# Load the 6,328-job LinkedIn dataset that will be used
# for semantic job matching and ranking.
# ============================================================

import pandas as pd
import os

JOB_FILE = "/content/LinkedIn_RDB_three.csv"

jobs_df = pd.read_csv(JOB_FILE)

print("✅ Job dataset loaded!")
print("📊 Number of jobs:", len(jobs_df))
print("📋 Number of columns:", len(jobs_df.columns))

print("\nColumns:")
print(jobs_df.columns.tolist())

print("\nFirst 3 jobs:")
display(jobs_df.head(3))

✅ Job dataset loaded!
📊 Number of jobs: 6328
📋 Number of columns: 15

Columns:
['job_id', 'title', 'description', 'pay_period', 'work_type', 'job_location', 'applies', 'remote_allowed', 'views', 'level', 'sponsored', 'compensation', 'job_domain', 'company_id', 'ben_pack_id']

First 3 jobs:


,job_id,title,description,pay_period,work_type,job_location,applies,remote_allowed,views,level,sponsored,compensation,job_domain,company_id,ben_pack_id
0,903408693,Office Associate,Provide clerical and administrative support to...,YEARLY,Full-time,"Albany, GA",5,False,49,NaN,True,"$42,000.00",Administration,3894635,1.0
1,1029078768,Registered Nurse (RN) Vaccinator,United Staffing Solutions is partnering with o...,HOURLY,Part-time,"Muskegon, MI",0,False,4,NaN,False,"$104,000.00",Healthcare,61469,1.0
2,1657978824,"REMOTE STEEL BUILDING SALES MAKE $1,000 TO $30...","REMOTE WORK FROM HOME $1,000 TO $10,000 COMMIS...",MONTHLY,Contract,"Texas, United States",0,True,0,NaN,False,"$144,000.00",Other,89350959,1.0


In [5]:
# ============================================================
# CELL 5 — BUILD CLEAN JOB DOCUMENTS
# WHY:
# Combine the important job fields into one rich text document
# for BGE semantic embedding and later LLM analysis.
# ============================================================

import pandas as pd

def build_job_text(row):
    parts = [
        f"Job Title: {row['title']}",
        f"Job Description: {row['description']}",
        f"Job Domain: {row['job_domain']}",
        f"Location: {row['job_location']}",
        f"Work Type: {row['work_type']}",
        f"Experience Level: {row['level']}"
    ]

    return "\n".join(
        str(x) for x in parts
        if pd.notna(x) and str(x).strip()
    )

jobs_df["job_text"] = jobs_df.apply(build_job_text, axis=1)

# Remove jobs without useful descriptions
jobs_df = jobs_df[
    jobs_df["description"].notna() &
    (jobs_df["description"].astype(str).str.strip() != "")
].reset_index(drop=True)

print("✅ Job documents created!")
print("📊 Usable jobs:", len(jobs_df))

print("\n📄 Sample job document:")
print(jobs_df.loc[0, "job_text"])

✅ Job documents created!
📊 Usable jobs: 6328

📄 Sample job document:
Job Title: Office Associate
Job Description: Provide clerical and administrative support to management as requestedAnswer phone calls and perform higher-level data entry relating to Customer ServiceProcess larger, more complicated customer orders and complaints and follow-up when necessaryProcess customer accounts and file documentsFollow communication procedures, guidelines, and policies, taking the extra mile to engage larger, more complex customersProvide accurate and timely information including product costs and standard costs for new productsAnalyze manufacturing processes in coordination with Plant Managers to improve decision-makingAlign projects with various accounting and customer experience needsOther duties as needed

Job Domain: Administration
Location: Albany, GA
Work Type: Full-time
Experience Level: nan


In [6]:
# ============================================================
# CELL 6 — GENERATE JOB EMBEDDINGS
# WHY:
# Convert every job document into a 1024-dimensional vector
# using BGE-large for semantic matching.
# ============================================================

print("⏳ Generating BGE embeddings for 6,328 jobs...")

job_embeddings = embedding_model.encode(
    jobs_df["job_text"].tolist(),
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("\n✅ Job embeddings generated!")
print("📐 Shape:", job_embeddings.shape)
print("🔢 Data type:", job_embeddings.dtype)
print("🖥️ Device:", embedding_model.device)

⏳ Generating BGE embeddings for 6,328 jobs...


Batches:   0%|          | 0/396 [00:00<?, ?it/s]


✅ Job embeddings generated!
📐 Shape: (6328, 1024)
🔢 Data type: float32
🖥️ Device: cuda:0


In [7]:
# ============================================================
# CELL 7 — CREATE FAISS VECTOR DATABASE
# WHY:
# FAISS enables fast semantic search across all 6,328
# embedded job descriptions.
# ============================================================

import faiss
import numpy as np

embedding_dimension = job_embeddings.shape[1]

job_index = faiss.IndexFlatIP(embedding_dimension)

job_index.add(
    np.ascontiguousarray(job_embeddings.astype("float32"))
)

print("✅ FAISS vector database created!")
print("📊 Total vectors:", job_index.ntotal)
print("📐 Vector dimension:", job_index.d)
print("🔍 Index type:", type(job_index).__name__)

✅ FAISS vector database created!
📊 Total vectors: 6328
📐 Vector dimension: 1024
🔍 Index type: IndexFlatIP


In [8]:
# ============================================================
# CELL 8 — UPLOAD & EXTRACT RESUME
# WHY:
# Extract the candidate's resume text so we can create
# a semantic resume embedding and compare it with jobs.
# ============================================================

import fitz
from google.colab import files

print("📄 Upload your latest resume PDF:")

uploaded_resume = files.upload()

resume_file = next(
    filename for filename in uploaded_resume
    if filename.lower().endswith(".pdf")
)

doc = fitz.open(resume_file)

resume_text = ""

for page in doc:
    resume_text += page.get_text()

doc.close()

print("\n✅ Resume extracted successfully!")
print("📄 File:", resume_file)
print("📝 Characters:", len(resume_text))

print("\n----- RESUME PREVIEW -----")
print(resume_text[:2000])

📄 Upload your latest resume PDF:


Saving HARSHA PRABHA VS RESUME.pdf to HARSHA PRABHA VS RESUME.pdf

✅ Resume extracted successfully!
📄 File: HARSHA PRABHA VS RESUME.pdf
📝 Characters: 3761

----- RESUME PREVIEW -----
Harsha Prabha V S
AI/ML Engineer
harshaprabha210@gmail.com
 
09344281602
 
LinkedIn
 
GitHub
 
SUMMARY
AI/ML Engineer and NLP Researcher with expertise in Machine Learning, Deep Learning, Natural Language Processing,
and Large Language Models. Published IEEE researcher building intelligent AI systems using Python, Transformers, and
Mamba State Space Models. Skilled in end-to-end AI applications, recommendation systems, and RAG-based solutions.
EDUCATION
St. Joseph's College of Engineering, B.Tech Information Technology  CGPA: 8.37
10/2022 – 04/2026 | Chennai
SKILLS
Languages: Python, SQL
Machine Learning & Deep Learning: Classification, Regression, Clustering, ANN, CNN, RNN, LSTM, Transformers, BERT, 
Mamba
NLP & Generative AI: NLP, Text Classification, Semantic Search, Embeddings, Sentence Transformers, R

In [9]:
# ============================================================
# CELL 9 — RESUME EMBEDDING + SEMANTIC JOB RETRIEVAL
# WHY:
# Convert the resume into the same BGE embedding space as
# the jobs, then retrieve the most semantically relevant jobs.
# ============================================================

resume_embedding = embedding_model.encode(
    [resume_text],
    normalize_embeddings=True,
    convert_to_numpy=True
).astype("float32")

print("✅ Resume embedding created!")
print("📐 Shape:", resume_embedding.shape)

# Retrieve top 20 semantic matches
TOP_K = 20

scores, indices = job_index.search(
    resume_embedding,
    TOP_K
)

matched_jobs = jobs_df.iloc[indices[0]].copy()

matched_jobs["semantic_score"] = scores[0] * 100

matched_jobs = matched_jobs.reset_index(drop=True)

print("\n🎯 TOP 20 SEMANTIC JOB MATCHES\n")

display(
    matched_jobs[
        [
            "title",
            "job_location",
            "work_type",
            "level",
            "job_domain",
            "semantic_score"
        ]
    ]
)

✅ Resume embedding created!
📐 Shape: (1, 1024)

🎯 TOP 20 SEMANTIC JOB MATCHES



,title,job_location,work_type,level,job_domain,semantic_score
0,Principal Machine Learning Engineer,United States,Full-time,Mid-Senior level,Information Technology,72.429016
1,"Software Engineer, Machine Learning - Reality ...","Boston, MA",Full-time,NaN,Engineering,70.955719
2,"Software Engineer, Machine Learning - Reality ...","Burlingame, CA",Full-time,NaN,Engineering,70.955719
3,ML Compiler,San Francisco Bay Area,Full-time,Mid-Senior level,Engineering,70.624069
4,Principal MLOps Engineer,United States,Full-time,Mid-Senior level,Engineering,70.467041
5,Machine Learning Scientist I,"Utah, United States",Full-time,Mid-Senior level,Other,70.170906
6,Machine Learning Scientist I,"Portland, OR",Full-time,Mid-Senior level,Other,70.170906
7,Machine Learning Engineer 5,"San Francisco, CA",Full-time,Entry level,Engineering,69.694794
8,Machine Learning Engineer 5,"San Jose, CA",Full-time,Entry level,Engineering,69.694794
9,Senior Machine Learning Engineer,United States,Full-time,Mid-Senior level,Engineering,69.290955


In [10]:
# ============================================================
# CELL 10 — LOAD QWEN 2.5 7B
# WHY:
# Qwen will analyze job experience requirements, seniority,
# required skills, and whether a fresher can realistically apply.
#
# IMPORTANT:
# We are NOT reinstalling torch/transformers here.
# ============================================================

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

print("🔍 Checking GPU...")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

LLM_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print("\n⏳ Loading Qwen tokenizer...")

llm_tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME
)

print("⏳ Loading Qwen 7B in 4-bit...")

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto"
)

llm_model.eval()

print("\n✅ Qwen 2.5 7B loaded successfully!")
print("📦 Quantization: 4-bit")
print("🖥️ Device:", llm_model.device)
print("💾 Model dtype:", llm_model.dtype)

🔍 Checking GPU...
GPU: Tesla T4
CUDA: 13.0

⏳ Loading Qwen tokenizer...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]


✅ Qwen 2.5 7B loaded successfully!
📦 Quantization: 4-bit
🖥️ Device: cuda:0
💾 Model dtype: torch.bfloat16


In [11]:
# ============================================================
# CELL 11 — LLM JOB ELIGIBILITY ENGINE
# WHY:
# BGE tells us which jobs are semantically similar.
# Qwen now checks whether those jobs are actually suitable
# for a fresher based on experience and seniority.
# ============================================================

import json
import re

def extract_json(text):
    text = re.sub(r"```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```\s*", "", text)

    match = re.search(r"\{.*\}", text, re.DOTALL)

    if not match:
        raise ValueError("❌ Qwen did not return valid JSON.")

    return json.loads(match.group(0))


def run_llm(prompt, max_new_tokens=500):

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert AI recruitment analyst. "
                "Analyze job descriptions accurately. "
                "Never invent requirements. "
                "Return ONLY valid JSON."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(
        text,
        return_tensors="pt"
    ).to(llm_model.device)

    with torch.no_grad():

        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=llm_tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    return llm_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )


def analyze_job(job_text):

    prompt = f"""
Analyze this job description for a fresh B.Tech graduate
targeting AI/ML roles.

JOB DESCRIPTION:
{job_text}

Return ONLY this JSON:

{{
    "role_type": "",
    "required_skills": [],
    "preferred_skills": [],
    "experience_years": 0,
    "seniority": "",
    "is_fresher_friendly": false,
    "reason": ""
}}

Rules:

1. Extract only information actually present.
2. experience_years must be a number.
3. If no experience requirement is mentioned, use 0.
4. seniority must be one of:
   Fresher, Junior, Mid-Level, Senior, Principal.
5. Set is_fresher_friendly=true only when a fresh graduate
   could realistically apply.
6. 5+ years experience is NOT fresher-friendly.
7. Principal/Senior roles are NOT fresher-friendly.
8. Return ONLY JSON.
"""

    raw = run_llm(
        prompt,
        max_new_tokens=500
    )

    return extract_json(raw)


print("✅ Qwen job eligibility engine is ready!")

✅ Qwen job eligibility engine is ready!


In [12]:
# ============================================================
# CELL 12 — TEST JOB ELIGIBILITY
# WHY:
# Test Qwen on one real job before running the model across
# all 20 retrieved jobs.
# ============================================================

test_job = matched_jobs.iloc[0]

print("🎯 TEST JOB")
print("Title:", test_job["title"])
print("Location:", test_job["job_location"])
print("Semantic Score:", round(test_job["semantic_score"], 2))

print("\n⏳ Analyzing job with Qwen...")

job_analysis = analyze_job(
    test_job["job_text"]
)

print("\n✅ QWEN ANALYSIS")
print(json.dumps(job_analysis, indent=4))

🎯 TEST JOB
Title: Principal Machine Learning Engineer 
Location: United States
Semantic Score: 72.43

⏳ Analyzing job with Qwen...

✅ QWEN ANALYSIS
{
    "role_type": "Principal Machine Learning Engineer",
    "required_skills": [
        "building and maintaining robust, large-scale Machine Learning and Data Pipelines",
        "in-depth experience with at least one machine learning framework (such as TensorFlow, PyTorch, Scikit-Learn etc.)",
        "exposure to applications of machine learning in healthcare"
    ],
    "preferred_skills": [
        "cross-functional collaboration",
        "architecting large systems with elegant interfaces that can scale easily"
    ],
    "experience_years": 8,
    "seniority": "Principal",
    "is_fresher_friendly": false,
    "reason": "The job description explicitly states 8+ years of experience and the role is titled 'Principal Machine Learning Engineer'."
}


In [13]:
# ============================================================
# CELL 13 — ANALYZE TOP 20 JOBS WITH QWEN
# WHY:
# Apply the eligibility engine to all semantically retrieved
# jobs and identify which ones are genuinely fresher-friendly.
# ============================================================

job_analyses = []

print("⏳ Analyzing 20 jobs with Qwen...\n")

for i, row in matched_jobs.iterrows():

    print(f"[{i+1}/20] {row['title']}")

    try:
        analysis = analyze_job(row["job_text"])

        job_analyses.append(analysis)

    except Exception as e:

        print("⚠️ Error:", e)

        job_analyses.append({
            "role_type": "",
            "required_skills": [],
            "preferred_skills": [],
            "experience_years": 0,
            "seniority": "",
            "is_fresher_friendly": False,
            "reason": "Analysis failed"
        })

print("\n✅ All 20 jobs analyzed!")

# Convert Qwen results to DataFrame
analysis_df = pd.DataFrame(job_analyses)

# Combine with semantic retrieval results
job_results = pd.concat(
    [
        matched_jobs.reset_index(drop=True),
        analysis_df.reset_index(drop=True)
    ],
    axis=1
)

print("\n📊 Fresher eligibility summary:")

display(
    job_results[
        [
            "title",
            "job_location",
            "semantic_score",
            "experience_years",
            "seniority",
            "is_fresher_friendly"
        ]
    ]
)

⏳ Analyzing 20 jobs with Qwen...

[1/20] Principal Machine Learning Engineer 
[2/20] Software Engineer, Machine Learning - Reality Labs
[3/20] Software Engineer, Machine Learning - Reality Labs
[4/20] ML Compiler 
[5/20] Principal MLOps Engineer
[6/20] Machine Learning Scientist I
[7/20] Machine Learning Scientist I
[8/20] Machine Learning Engineer 5
[9/20] Machine Learning Engineer 5
[10/20] Senior Machine Learning Engineer
[11/20] Senior ML Specialist Solutions Architect, ML Service Solution Architects
[12/20] Machine Learning Engineer, Senior
[13/20] AIML - Senior Data Infrastructure Software Engineer, Machine Learning Platform and Technology
[14/20] Machine Learning Engineer - Notifications
[15/20] Director, Machine Learning Platform
[16/20] Frontend Web Developer (Machine Learning)
[17/20] Research Scientist
[18/20] AI Consultant, Google Cloud
[19/20] AI Consultant, Google Cloud
[20/20] AI Consultant, Google Cloud

✅ All 20 jobs analyzed!

📊 Fresher eligibility summary:


,title,job_location,semantic_score,experience_years,seniority,is_fresher_friendly
0,Principal Machine Learning Engineer,United States,72.429016,8,Principal,False
1,"Software Engineer, Machine Learning - Reality ...","Boston, MA",70.955719,6,Senior,False
2,"Software Engineer, Machine Learning - Reality ...","Burlingame, CA",70.955719,6,Senior,False
3,ML Compiler,San Francisco Bay Area,70.624069,0,Mid-Level,False
4,Principal MLOps Engineer,United States,70.467041,4,Principal,False
5,Machine Learning Scientist I,"Utah, United States",70.170906,0,Mid-Level,True
6,Machine Learning Scientist I,"Portland, OR",70.170906,0,Mid-Level,True
7,Machine Learning Engineer 5,"San Francisco, CA",69.694794,5,Mid-Level,False
8,Machine Learning Engineer 5,"San Jose, CA",69.694794,5,Mid-Level,False
9,Senior Machine Learning Engineer,United States,69.290955,2,Senior,False


In [14]:
# ============================================================
# CELL 14 — FILTER ELIGIBLE JOBS
# WHY:
# Remove jobs that require significant experience or senior/
# principal-level roles before calculating the final match.
# ============================================================

eligible_jobs = job_results[
    job_results["is_fresher_friendly"] == True
].copy()

eligible_jobs = eligible_jobs.sort_values(
    by="semantic_score",
    ascending=False
).reset_index(drop=True)

print("✅ Fresher-friendly jobs found:", len(eligible_jobs))

display(
    eligible_jobs[
        [
            "title",
            "job_location",
            "work_type",
            "semantic_score",
            "experience_years",
            "seniority",
            "required_skills",
            "preferred_skills",
            "reason"
        ]
    ]
)

✅ Fresher-friendly jobs found: 2


,title,job_location,work_type,semantic_score,experience_years,seniority,required_skills,preferred_skills,reason
0,Machine Learning Scientist I,"Utah, United States",Full-time,70.170906,0,Mid-Level,[Strong mathematical foundation and understand...,[Academic degree (master's or PhD preferred) i...,The job description does not specify a minimum...
1,Machine Learning Scientist I,"Portland, OR",Full-time,70.170906,0,Mid-Level,[Strong mathematical foundation and understand...,"[Academic degree in Data Science, Computer Sci...",The job description does not specify a minimum...


In [15]:
# ============================================================
# CELL 15 — EXTRACT CANDIDATE SKILLS
# WHY:
# Compare the skills explicitly present in the resume against
# the requirements extracted from each eligible job.
# ============================================================

import re

SKILL_LIBRARY = [
    "Python", "SQL", "Machine Learning", "Deep Learning",
    "Classification", "Regression", "Clustering",
    "ANN", "CNN", "RNN", "LSTM",
    "Transformers", "BERT", "Mamba",
    "NLP", "Text Classification", "Semantic Search",
    "Embeddings", "Sentence Transformers",
    "RAG", "LLMs", "Prompt Engineering",
    "NumPy", "Pandas", "Matplotlib", "Seaborn",
    "PyTorch", "LangChain", "LangGraph",
    "FastAPI", "Streamlit",
    "HTML", "CSS", "JavaScript", "React.js"
]

resume_lower = resume_text.lower()

candidate_skills = []

for skill in SKILL_LIBRARY:
    if skill.lower() in resume_lower:
        candidate_skills.append(skill)

candidate_skills = sorted(
    set(candidate_skills)
)

print("✅ Candidate skills extracted!")
print("📊 Total skills:", len(candidate_skills))

print("\n🎓 Candidate skill profile:")
for skill in candidate_skills:
    print(" •", skill)

✅ Candidate skills extracted!
📊 Total skills: 33

🎓 Candidate skill profile:
 • ANN
 • BERT
 • CNN
 • CSS
 • Classification
 • Clustering
 • Deep Learning
 • Embeddings
 • FastAPI
 • HTML
 • JavaScript
 • LLMs
 • LSTM
 • Machine Learning
 • Mamba
 • Matplotlib
 • NLP
 • NumPy
 • Pandas
 • Prompt Engineering
 • PyTorch
 • Python
 • RAG
 • RNN
 • React.js
 • Regression
 • SQL
 • Seaborn
 • Semantic Search
 • Sentence Transformers
 • Streamlit
 • Text Classification
 • Transformers


In [16]:
# ============================================================
# CELL 16 — SKILL MATCH SCORING
# WHY:
# Compare the candidate's actual resume skills with the
# requirements extracted by Qwen for each job.
# ============================================================

def normalize_skill(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9+#. ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


candidate_skill_text = [
    normalize_skill(skill)
    for skill in candidate_skills
]


def calculate_skill_match(required_skills):

    if not required_skills:
        return 0.0, []

    matched = []

    for requirement in required_skills:

        requirement_text = normalize_skill(requirement)

        # Check whether any candidate skill appears
        # meaningfully inside the requirement.
        for candidate_skill in candidate_skill_text:

            if (
                candidate_skill in requirement_text
                or requirement_text in candidate_skill
            ):
                matched.append(candidate_skill)
                break

    matched = sorted(set(matched))

    score = (
        len(matched) / len(required_skills)
    ) * 100

    return score, matched


skill_scores = []
matched_skill_lists = []

for _, row in eligible_jobs.iterrows():

    score, matched = calculate_skill_match(
        row["required_skills"]
    )

    skill_scores.append(score)
    matched_skill_lists.append(matched)


eligible_jobs["skill_match_score"] = skill_scores
eligible_jobs["matched_skills"] = matched_skill_lists

print("✅ Skill matching completed!")

display(
    eligible_jobs[
        [
            "title",
            "semantic_score",
            "skill_match_score",
            "matched_skills",
            "required_skills"
        ]
    ]
)

✅ Skill matching completed!


,title,semantic_score,skill_match_score,matched_skills,required_skills
0,Machine Learning Scientist I,70.170906,50.0,"[deep learning, machine learning, python]",[Strong mathematical foundation and understand...
1,Machine Learning Scientist I,70.170906,37.5,"[deep learning, machine learning, python]",[Strong mathematical foundation and understand...


In [17]:
# ============================================================
# CELL 17 — FINAL JOB MATCH SCORE
# WHY:
# Combine semantic relevance and skill compatibility into
# one practical ranking score for the candidate.
# ============================================================

# Weighted scoring:
# 60% = semantic similarity
# 40% = required skill match

eligible_jobs["final_match_score"] = (
    eligible_jobs["semantic_score"] * 0.60
    + eligible_jobs["skill_match_score"] * 0.40
)

# Rank highest match first
eligible_jobs = eligible_jobs.sort_values(
    by="final_match_score",
    ascending=False
).reset_index(drop=True)

print("✅ Final job ranking calculated!")

display(
    eligible_jobs[
        [
            "title",
            "job_location",
            "work_type",
            "semantic_score",
            "skill_match_score",
            "final_match_score",
            "matched_skills"
        ]
    ]
)

✅ Final job ranking calculated!


,title,job_location,work_type,semantic_score,skill_match_score,final_match_score,matched_skills
0,Machine Learning Scientist I,"Utah, United States",Full-time,70.170906,50.0,62.102547,"[deep learning, machine learning, python]"
1,Machine Learning Scientist I,"Portland, OR",Full-time,70.170906,37.5,57.102547,"[deep learning, machine learning, python]"


In [18]:
# ============================================================
# CELL 18 — LOCATION PREFERENCE SCORE
# WHY:
# Give higher ranking to preferred locations while keeping
# international/other-location jobs clearly identifiable.
# ============================================================

PREFERRED_LOCATIONS = [
    "Chennai",
    "Bangalore",
    "Bengaluru",
    "Hyderabad",
    "Coimbatore"
]

def calculate_location_score(location):

    location = str(location).lower()

    for preferred in PREFERRED_LOCATIONS:
        if preferred.lower() in location:
            return 100.0

    # Remote jobs receive a moderate preference
    # because they can potentially be applied for from India.
    if "remote" in location:
        return 70.0

    return 0.0


eligible_jobs["location_score"] = (
    eligible_jobs["job_location"]
    .apply(calculate_location_score)
)

# Final weighted score:
# 50% Semantic relevance
# 35% Skill match
# 15% Location preference

eligible_jobs["final_match_score"] = (
    eligible_jobs["semantic_score"] * 0.50
    + eligible_jobs["skill_match_score"] * 0.35
    + eligible_jobs["location_score"] * 0.15
)

eligible_jobs = eligible_jobs.sort_values(
    by="final_match_score",
    ascending=False
).reset_index(drop=True)

print("✅ Location-aware ranking completed!")

display(
    eligible_jobs[
        [
            "title",
            "job_location",
            "semantic_score",
            "skill_match_score",
            "location_score",
            "final_match_score"
        ]
    ]
)

✅ Location-aware ranking completed!


,title,job_location,semantic_score,skill_match_score,location_score,final_match_score
0,Machine Learning Scientist I,"Utah, United States",70.170906,50.0,0.0,52.585453
1,Machine Learning Scientist I,"Portland, OR",70.170906,37.5,0.0,48.210453


In [19]:
# ============================================================
# CELL 19 — EXPAND SEMANTIC SEARCH
# WHY:
# Top-20 retrieval contains mostly senior jobs.
# Search more jobs so we can discover additional
# fresher-friendly AI/ML opportunities.
# ============================================================

EXPANDED_K = 100

expanded_scores, expanded_indices = job_index.search(
    resume_embedding,
    EXPANDED_K
)

expanded_jobs = jobs_df.iloc[
    expanded_indices[0]
].copy().reset_index(drop=True)

expanded_jobs["semantic_score"] = (
    expanded_scores[0] * 100
)

print("✅ Expanded semantic search completed!")
print("📊 Jobs retrieved:", len(expanded_jobs))

display(
    expanded_jobs[
        [
            "title",
            "job_location",
            "work_type",
            "level",
            "job_domain",
            "semantic_score"
        ]
    ].head(30)
)

✅ Expanded semantic search completed!
📊 Jobs retrieved: 100


,title,job_location,work_type,level,job_domain,semantic_score
0,Principal Machine Learning Engineer,United States,Full-time,Mid-Senior level,Information Technology,72.429016
1,"Software Engineer, Machine Learning - Reality ...","Boston, MA",Full-time,NaN,Engineering,70.955719
2,"Software Engineer, Machine Learning - Reality ...","Burlingame, CA",Full-time,NaN,Engineering,70.955719
3,ML Compiler,San Francisco Bay Area,Full-time,Mid-Senior level,Engineering,70.624069
4,Principal MLOps Engineer,United States,Full-time,Mid-Senior level,Engineering,70.467041
5,Machine Learning Scientist I,"Utah, United States",Full-time,Mid-Senior level,Other,70.170906
6,Machine Learning Scientist I,"Portland, OR",Full-time,Mid-Senior level,Other,70.170906
7,Machine Learning Engineer 5,"San Francisco, CA",Full-time,Entry level,Engineering,69.694794
8,Machine Learning Engineer 5,"San Jose, CA",Full-time,Entry level,Engineering,69.694794
9,Senior Machine Learning Engineer,United States,Full-time,Mid-Senior level,Engineering,69.290955


In [20]:
# ============================================================
# CELL 20 — PRE-FILTER JOBS
# WHY:
# Remove obvious Senior/Principal/Director/Lead roles before
# sending jobs to Qwen. This saves GPU time and improves ranking.
# ============================================================

EXCLUDED_LEVELS = [
    "mid-senior level",
    "director",
    "executive"
]

EXCLUDED_TITLE_WORDS = [
    "senior",
    "principal",
    "director",
    "lead",
    "staff",
    "chief",
    "head"
]

def is_potential_entry_job(row):

    level = str(row["level"]).lower()
    title = str(row["title"]).lower()

    # Reject clearly senior levels
    if level in EXCLUDED_LEVELS:
        return False

    # Reject clearly senior titles
    for word in EXCLUDED_TITLE_WORDS:
        if word in title:
            return False

    return True


candidate_jobs = expanded_jobs[
    expanded_jobs.apply(is_potential_entry_job, axis=1)
].copy()

candidate_jobs = candidate_jobs.reset_index(drop=True)

print("✅ Pre-filter completed!")
print("📊 Jobs before filtering:", len(expanded_jobs))
print("🎯 Potential entry-level jobs:", len(candidate_jobs))

display(
    candidate_jobs[
        [
            "title",
            "job_location",
            "work_type",
            "level",
            "job_domain",
            "semantic_score"
        ]
    ].head(30)
)

✅ Pre-filter completed!
📊 Jobs before filtering: 100
🎯 Potential entry-level jobs: 46


,title,job_location,work_type,level,job_domain,semantic_score
0,"Software Engineer, Machine Learning - Reality ...","Boston, MA",Full-time,NaN,Engineering,70.955719
1,"Software Engineer, Machine Learning - Reality ...","Burlingame, CA",Full-time,NaN,Engineering,70.955719
2,Machine Learning Engineer 5,"San Francisco, CA",Full-time,Entry level,Engineering,69.694794
3,Machine Learning Engineer 5,"San Jose, CA",Full-time,Entry level,Engineering,69.694794
4,"AI Consultant, Google Cloud","Chicago, IL",Full-time,NaN,Project Management,67.764893
5,"AI Consultant, Google Cloud","Irvine, CA",Full-time,NaN,Project Management,67.764893
6,"AI Consultant, Google Cloud","Los Angeles, CA",Full-time,NaN,Project Management,67.764893
7,"AI Consultant, Google Cloud","Seattle, WA",Full-time,NaN,Project Management,67.764893
8,Automation Engineer,"Plattsburgh, NY",Full-time,Associate,Engineering,67.753304
9,"Hardware Architect, Machine Learning, Technica...","Sunnyvale, CA",Full-time,NaN,Other,67.289909


In [21]:
# ============================================================
# CELL 21 — QWEN ANALYSIS OF ENTRY-LEVEL CANDIDATES
# WHY:
# The dataset's "Entry level" label alone is not enough.
# Qwen will inspect the actual description for experience,
# skills, seniority, and fresher suitability.
# ============================================================

qwen_results = []

print("⏳ Analyzing potential entry-level jobs with Qwen...\n")

for i, row in candidate_jobs.iterrows():

    print(f"[{i+1}/{len(candidate_jobs)}] {row['title']}")

    try:
        result = analyze_job(row["job_text"])
        qwen_results.append(result)

    except Exception as e:

        print("⚠️ Error:", e)

        qwen_results.append({
            "role_type": "",
            "required_skills": [],
            "preferred_skills": [],
            "experience_years": 99,
            "seniority": "Senior",
            "is_fresher_friendly": False,
            "reason": "Qwen analysis failed"
        })

qwen_df = pd.DataFrame(qwen_results)

candidate_jobs_qwen = pd.concat(
    [
        candidate_jobs.reset_index(drop=True),
        qwen_df.reset_index(drop=True)
    ],
    axis=1
)

print("\n✅ Qwen analysis completed!")

display(
    candidate_jobs_qwen[
        [
            "title",
            "job_location",
            "semantic_score",
            "experience_years",
            "seniority",
            "is_fresher_friendly"
        ]
    ]
)

⏳ Analyzing potential entry-level jobs with Qwen...

[1/46] Software Engineer, Machine Learning - Reality Labs
[2/46] Software Engineer, Machine Learning - Reality Labs
[3/46] Machine Learning Engineer 5
[4/46] Machine Learning Engineer 5
[5/46] AI Consultant, Google Cloud
[6/46] AI Consultant, Google Cloud
[7/46] AI Consultant, Google Cloud
[8/46] AI Consultant, Google Cloud
[9/46] Automation Engineer
[10/46] Hardware Architect, Machine Learning, Technical Infrastructure
[11/46] Data Scientist
[12/46] Research Internship
[13/46] Natural Language Processing (NLP) Data Scientist.
[14/46] Research Associate III
[15/46] Research Associate
[16/46] Full Stack Software Engineer (Java)
[17/46] Data Engineer, Google Nest
[18/46] Software Engineer III, Hardware/Software Co-Design
[19/46] Software Engineer III, Hardware/Software Co-Design
[20/46] Software Engineer III, Hardware/Software Co-Design
[21/46] Software Engineer III, Full Stack, Google Cloud Application Modernization
[22/46] Software E

,title,job_location,semantic_score,experience_years,seniority,is_fresher_friendly
0,"Software Engineer, Machine Learning - Reality ...","Boston, MA",70.955719,6,Senior,False
1,"Software Engineer, Machine Learning - Reality ...","Burlingame, CA",70.955719,6,Senior,False
2,Machine Learning Engineer 5,"San Francisco, CA",69.694794,5,Mid-Level,False
3,Machine Learning Engineer 5,"San Jose, CA",69.694794,5,Mid-Level,False
4,"AI Consultant, Google Cloud","Chicago, IL",67.764893,10,Senior,False
5,"AI Consultant, Google Cloud","Irvine, CA",67.764893,10,Senior,False
6,"AI Consultant, Google Cloud","Los Angeles, CA",67.764893,10,Senior,False
7,"AI Consultant, Google Cloud","Seattle, WA",67.764893,10,Senior,False
8,Automation Engineer,"Plattsburgh, NY",67.753304,5,Mid-Level,False
9,"Hardware Architect, Machine Learning, Technica...","Sunnyvale, CA",67.289909,8,Senior,False


In [22]:
# ============================================================
# CELL 22 — FINAL FRESHER ELIGIBILITY FILTER
# WHY:
# Keep only jobs that Qwen considers realistically suitable
# for a fresh graduate.
# ============================================================

fresher_jobs = candidate_jobs_qwen[
    candidate_jobs_qwen["is_fresher_friendly"] == True
].copy()

fresher_jobs = fresher_jobs.sort_values(
    by="semantic_score",
    ascending=False
).reset_index(drop=True)

print("✅ Final fresher-friendly jobs:", len(fresher_jobs))

display(
    fresher_jobs[
        [
            "title",
            "job_location",
            "work_type",
            "semantic_score",
            "experience_years",
            "seniority",
            "is_fresher_friendly",
            "reason"
        ]
    ]
)

✅ Final fresher-friendly jobs: 9


,title,job_location,work_type,semantic_score,experience_years,seniority,is_fresher_friendly,reason
0,Research Internship,United States,Internship,66.798431,0,Fresher,True,The job description explicitly mentions inviti...
1,Research Associate,"San Mateo County, CA",Contract,64.390022,0,Fresher,True,The job description is for a Research Associat...
2,"Data Engineer, Google Nest","Mountain View, CA",Full-time,64.228401,0,Fresher,True,The job description does not specify any minim...
3,Bioinformatician I/II,"Seattle, WA",Full-time,63.208111,0,Fresher,True,The job description does not specify any exper...
4,Emerging Technology Engineer,"Kalamazoo, MI",Full-time,63.191246,0,Fresher,True,The job description does not specify any exper...
5,Validation Engineer,"Raynham, MA",Full-time,63.088070,0,Fresher,True,The job description does not specify any exper...
6,Full-Stack Software Engineer,"San Francisco, CA",Full-time,62.978561,0,Junior,True,The job description does not specify a minimum...
7,Scientist,"New Brunswick, NJ",Contract,62.562634,0,Fresher,True,The role requires a B.Tech degree and does not...
8,Software Engineer - Database Engineering (San ...,"San Mateo, CA",Full-time,62.495667,2,Junior,True,The job description does not explicitly requir...


In [23]:
# ============================================================
# CELL 23 — FINAL SKILL MATCH
# WHY:
# Compare Harsha's actual resume skills against the
# Qwen-extracted required skills for the 9 eligible jobs.
# ============================================================

skill_scores = []
matched_skill_lists = []

for _, row in fresher_jobs.iterrows():

    score, matched = calculate_skill_match(
        row["required_skills"]
    )

    skill_scores.append(score)
    matched_skill_lists.append(matched)

fresher_jobs["skill_match_score"] = skill_scores
fresher_jobs["matched_skills"] = matched_skill_lists

print("✅ Final skill matching completed!")

display(
    fresher_jobs[
        [
            "title",
            "job_location",
            "semantic_score",
            "skill_match_score",
            "matched_skills",
            "required_skills"
        ]
    ]
)

✅ Final skill matching completed!


,title,job_location,semantic_score,skill_match_score,matched_skills,required_skills
0,Research Internship,United States,66.798431,50.000000,[machine learning],"[Strong software engineering skills, Research ..."
1,Research Associate,"San Mateo County, CA",64.390022,0.000000,[],"[Execute defined procedures, Analyze samples u..."
2,"Data Engineer, Google Nest","Mountain View, CA",64.228401,66.666667,"[python, sql]",[Bachelor's degree in Computer Science or rela...
3,Bioinformatician I/II,"Seattle, WA",63.208111,0.000000,[],"[analysis of high-throughput data sets, transc..."
4,Emerging Technology Engineer,"Kalamazoo, MI",63.191246,100.000000,[python],"[Programming languages such as Python, Java, o..."
5,Validation Engineer,"Raynham, MA",63.088070,0.000000,[],"[Review and approve documents, Assist in devel..."
6,Full-Stack Software Engineer,"San Francisco, CA",62.978561,0.000000,[],"[writing professional production code, buildin..."
7,Scientist,"New Brunswick, NJ",62.562634,0.000000,[],"[Execute hands-on pilot scale operations, Perf..."
8,Software Engineer - Database Engineering (San ...,"San Mateo, CA",62.495667,0.000000,[],"[Fluency in Java or C++, Familiarity with deve..."


In [24]:
# ============================================================
# CELL 24 — FINAL AI JOB RANKING
# WHY:
# Combine semantic relevance and actual skill compatibility.
# Fresher eligibility is already enforced as a hard filter.
# ============================================================

fresher_jobs["final_match_score"] = (
    fresher_jobs["semantic_score"] * 0.55
    + fresher_jobs["skill_match_score"] * 0.45
)

fresher_jobs = fresher_jobs.sort_values(
    by="final_match_score",
    ascending=False
).reset_index(drop=True)

print("✅ FINAL AI JOB RANKING COMPLETED!")

display(
    fresher_jobs[
        [
            "title",
            "job_location",
            "work_type",
            "semantic_score",
            "skill_match_score",
            "final_match_score",
            "matched_skills"
        ]
    ]
)

✅ FINAL AI JOB RANKING COMPLETED!


,title,job_location,work_type,semantic_score,skill_match_score,final_match_score,matched_skills
0,Emerging Technology Engineer,"Kalamazoo, MI",Full-time,63.191246,100.000000,79.755184,[python]
1,"Data Engineer, Google Nest","Mountain View, CA",Full-time,64.228401,66.666667,65.325623,"[python, sql]"
2,Research Internship,United States,Internship,66.798431,50.000000,59.239140,[machine learning]
3,Research Associate,"San Mateo County, CA",Contract,64.390022,0.000000,35.414513,[]
4,Bioinformatician I/II,"Seattle, WA",Full-time,63.208111,0.000000,34.764462,[]
5,Validation Engineer,"Raynham, MA",Full-time,63.088070,0.000000,34.698441,[]
6,Full-Stack Software Engineer,"San Francisco, CA",Full-time,62.978561,0.000000,34.638210,[]
7,Scientist,"New Brunswick, NJ",Contract,62.562634,0.000000,34.409451,[]
8,Software Engineer - Database Engineering (San ...,"San Mateo, CA",Full-time,62.495667,0.000000,34.372616,[]


In [25]:
# ============================================================
# CELL 25 — AI/ML ROLE RELEVANCE
# WHY:
# Prevent a generic Python/software role from outranking a
# genuinely relevant AI/ML/NLP/Data Science opportunity.
# ============================================================

AI_ML_KEYWORDS = [
    "machine learning",
    "artificial intelligence",
    "ai",
    "ml",
    "deep learning",
    "nlp",
    "natural language",
    "data scientist",
    "data science",
    "research",
    "computer vision",
    "pytorch",
    "tensorflow",
    "llm",
    "generative ai",
    "recommendation"
]

def calculate_role_relevance(row):

    text = (
        str(row["title"]) + " " +
        str(row["job_domain"]) + " " +
        str(row["job_text"])
    ).lower()

    matches = [
        keyword
        for keyword in AI_ML_KEYWORDS
        if keyword in text
    ]

    # Cap the score at 100
    score = min(len(set(matches)) * 10, 100)

    return score


fresher_jobs["role_relevance_score"] = (
    fresher_jobs.apply(
        calculate_role_relevance,
        axis=1
    )
)

# New ranking:
# 45% Semantic relevance
# 35% Skill match
# 20% AI/ML role relevance

fresher_jobs["final_match_score"] = (
    fresher_jobs["semantic_score"] * 0.45
    + fresher_jobs["skill_match_score"] * 0.35
    + fresher_jobs["role_relevance_score"] * 0.20
)

fresher_jobs = fresher_jobs.sort_values(
    by="final_match_score",
    ascending=False
).reset_index(drop=True)

print("✅ AI/ML-aware ranking completed!")

display(
    fresher_jobs[
        [
            "title",
            "job_location",
            "semantic_score",
            "skill_match_score",
            "role_relevance_score",
            "final_match_score"
        ]
    ]
)

✅ AI/ML-aware ranking completed!


,title,job_location,semantic_score,skill_match_score,role_relevance_score,final_match_score
0,Emerging Technology Engineer,"Kalamazoo, MI",63.191246,100.000000,70,77.436060
1,Research Internship,United States,66.798431,50.000000,100,67.559294
2,"Data Engineer, Google Nest","Mountain View, CA",64.228401,66.666667,40,60.236114
3,Research Associate,"San Mateo County, CA",64.390022,0.000000,20,32.975510
4,Bioinformatician I/II,"Seattle, WA",63.208111,0.000000,20,32.443649
5,Scientist,"New Brunswick, NJ",62.562634,0.000000,20,32.153185
6,Validation Engineer,"Raynham, MA",63.088070,0.000000,10,30.389631
7,Full-Stack Software Engineer,"San Francisco, CA",62.978561,0.000000,10,30.340351
8,Software Engineer - Database Engineering (San ...,"San Mateo, CA",62.495667,0.000000,10,30.123049


In [26]:
# ============================================================
# CELL 26 — QWEN-BASED ROLE RELEVANCE
# WHY:
# Use Qwen's understanding of the job instead of simple
# keyword counting to determine how relevant the role is
# to the candidate's AI/ML career target.
# ============================================================

AI_ML_ROLE_TYPES = [
    "machine learning",
    "artificial intelligence",
    "ai engineer",
    "ml engineer",
    "data scientist",
    "research scientist",
    "nlp",
    "deep learning",
    "computer vision",
    "llm",
    "research"
]

def calculate_qwen_role_relevance(row):

    role = str(row["role_type"]).lower()
    skills = " ".join(
        str(skill).lower()
        for skill in row["required_skills"]
    )

    combined = role + " " + skills

    strong_matches = [
        keyword
        for keyword in AI_ML_ROLE_TYPES
        if keyword in combined
    ]

    if len(strong_matches) >= 2:
        return 100.0
    elif len(strong_matches) == 1:
        return 80.0
    else:
        return 20.0


fresher_jobs["qwen_role_relevance"] = (
    fresher_jobs.apply(
        calculate_qwen_role_relevance,
        axis=1
    )
)

# Final score:
# 45% semantic relevance
# 35% skill compatibility
# 20% Qwen role relevance

fresher_jobs["final_match_score"] = (
    fresher_jobs["semantic_score"] * 0.45
    + fresher_jobs["skill_match_score"] * 0.35
    + fresher_jobs["qwen_role_relevance"] * 0.20
)

fresher_jobs = fresher_jobs.sort_values(
    by="final_match_score",
    ascending=False
).reset_index(drop=True)

print("✅ Qwen-based final ranking completed!")

display(
    fresher_jobs[
        [
            "title",
            "job_location",
            "role_type",
            "semantic_score",
            "skill_match_score",
            "qwen_role_relevance",
            "final_match_score"
        ]
    ]
)

✅ Qwen-based final ranking completed!


,title,job_location,role_type,semantic_score,skill_match_score,qwen_role_relevance,final_match_score
0,Research Internship,United States,Research Internship,66.798431,50.000000,100.0,67.559294
1,Emerging Technology Engineer,"Kalamazoo, MI",Emerging Technology Engineer,63.191246,100.000000,20.0,67.436060
2,"Data Engineer, Google Nest","Mountain View, CA",Data Engineer,64.228401,66.666667,20.0,56.236114
3,Research Associate,"San Mateo County, CA",Research,64.390022,0.000000,80.0,44.975510
4,Bioinformatician I/II,"Seattle, WA",Bioinformatician I/II,63.208111,0.000000,20.0,32.443649
5,Validation Engineer,"Raynham, MA",,63.088070,0.000000,20.0,32.389631
6,Full-Stack Software Engineer,"San Francisco, CA",Full-Stack Software Engineer,62.978561,0.000000,20.0,32.340351
7,Scientist,"New Brunswick, NJ",Assistant Scientist,62.562634,0.000000,20.0,32.153185
8,Software Engineer - Database Engineering (San ...,"San Mateo, CA",Software Engineer,62.495667,0.000000,20.0,32.123049


In [27]:
# ============================================================
# CELL 27 — FINAL CANDIDATE FIT SCORE
# WHY:
# Combine:
#   1. Semantic job relevance
#   2. Skill compatibility
#   3. AI/ML role relevance
#   4. Fresher/experience fit
#   5. Preferred location
# ============================================================

PREFERRED_LOCATIONS = [
    "chennai",
    "bangalore",
    "bengaluru",
    "hyderabad",
    "coimbatore"
]

def calculate_location_score(location):

    location = str(location).lower()

    for preferred in PREFERRED_LOCATIONS:
        if preferred in location:
            return 100.0

    if "remote" in location:
        return 70.0

    return 0.0


def calculate_experience_fit(row):

    experience = float(row["experience_years"])

    # Fresher / no minimum experience
    if experience == 0:
        return 100.0

    # Candidate has internship experience but is still a fresher
    if experience <= 1:
        return 80.0

    # 2+ years is not an ideal fresher match
    return 20.0


fresher_jobs["location_score"] = (
    fresher_jobs["job_location"]
    .apply(calculate_location_score)
)

fresher_jobs["experience_fit_score"] = (
    fresher_jobs.apply(
        calculate_experience_fit,
        axis=1
    )
)

# Final weights
# 35% Semantic relevance
# 30% Skill compatibility
# 20% AI/ML role relevance
# 10% Experience fit
# 5% Location preference

fresher_jobs["final_candidate_score"] = (
    fresher_jobs["semantic_score"] * 0.35
    + fresher_jobs["skill_match_score"] * 0.30
    + fresher_jobs["qwen_role_relevance"] * 0.20
    + fresher_jobs["experience_fit_score"] * 0.10
    + fresher_jobs["location_score"] * 0.05
)

fresher_jobs = fresher_jobs.sort_values(
    by="final_candidate_score",
    ascending=False
).reset_index(drop=True)

print("✅ FINAL CANDIDATE-JOB SCORING COMPLETED!")

display(
    fresher_jobs[
        [
            "title",
            "job_location",
            "work_type",
            "semantic_score",
            "skill_match_score",
            "qwen_role_relevance",
            "experience_fit_score",
            "location_score",
            "final_candidate_score"
        ]
    ]
)

✅ FINAL CANDIDATE-JOB SCORING COMPLETED!


,title,job_location,work_type,semantic_score,skill_match_score,qwen_role_relevance,experience_fit_score,location_score,final_candidate_score
0,Research Internship,United States,Internship,66.798431,50.000000,100.0,100.0,0.0,68.379450
1,Emerging Technology Engineer,"Kalamazoo, MI",Full-time,63.191246,100.000000,20.0,100.0,0.0,66.116936
2,"Data Engineer, Google Nest","Mountain View, CA",Full-time,64.228401,66.666667,20.0,100.0,0.0,56.479940
3,Research Associate,"San Mateo County, CA",Contract,64.390022,0.000000,80.0,100.0,0.0,48.536507
4,Bioinformatician I/II,"Seattle, WA",Full-time,63.208111,0.000000,20.0,100.0,0.0,36.122839
5,Validation Engineer,"Raynham, MA",Full-time,63.088070,0.000000,20.0,100.0,0.0,36.080824
6,Full-Stack Software Engineer,"San Francisco, CA",Full-time,62.978561,0.000000,20.0,100.0,0.0,36.042496
7,Scientist,"New Brunswick, NJ",Contract,62.562634,0.000000,20.0,100.0,0.0,35.896921
8,Software Engineer - Database Engineering (San ...,"San Mateo, CA",Full-time,62.495667,0.000000,20.0,20.0,0.0,27.873484


In [28]:
# ============================================================
# CELL 28 — AI JOB RECOMMENDATION EXPLANATION
# WHY:
# Qwen explains why each job matches the candidate,
# identifies strengths, missing skills, and recommends
# whether the candidate should apply.
# ============================================================

def generate_job_explanation(row):

    prompt = f"""
You are an expert AI career advisor.

Candidate:
{resume_text}

Job Title:
{row['title']}

Job Location:
{row['job_location']}

Job Description:
{row['job_text']}

Candidate Scores:
Semantic Score: {row['semantic_score']:.2f}
Skill Match: {row['skill_match_score']:.2f}
AI/ML Relevance: {row['qwen_role_relevance']:.2f}
Experience Fit: {row['experience_fit_score']:.2f}

Return ONLY valid JSON:

{{
  "why_match": "short explanation",
  "matching_skills": ["skill1", "skill2"],
  "missing_skills": ["skill1", "skill2"],
  "experience_fit": "short explanation",
  "application_recommendation": "Strong Apply / Apply / Low Priority",
  "reason": "short final reason"
}}
"""

    result = run_llm(prompt)

    return extract_json(result)


# Generate explanations for top 5 jobs
top_jobs = fresher_jobs.head(5).copy()

explanations = []

for _, row in top_jobs.iterrows():

    print(f"🔍 Analyzing: {row['title']}")

    explanation = generate_job_explanation(row)

    explanations.append(explanation)

top_jobs["ai_explanation"] = explanations

print("\n✅ AI JOB EXPLANATIONS GENERATED!")

🔍 Analyzing: Research Internship
🔍 Analyzing: Emerging Technology Engineer
🔍 Analyzing: Data Engineer, Google Nest
🔍 Analyzing: Research Associate
🔍 Analyzing: Bioinformatician I/II

✅ AI JOB EXPLANATIONS GENERATED!


In [29]:
# ============================================================
# CELL 29 — SKILL GAP ANALYSIS
# WHY:
# Combine the AI explanations from the top jobs and identify
# the most important missing skills for the candidate.
# ============================================================

from collections import Counter

all_missing_skills = []

for explanation in top_jobs["ai_explanation"]:

    if isinstance(explanation, dict):

        missing = explanation.get("missing_skills", [])

        if isinstance(missing, list):
            all_missing_skills.extend(missing)


# Normalize skill names
normalized_missing = [
    str(skill).strip().lower()
    for skill in all_missing_skills
    if str(skill).strip()
]


# Count how frequently each missing skill appears
skill_gap_counts = Counter(normalized_missing)


skill_gap_df = pd.DataFrame(
    skill_gap_counts.items(),
    columns=["skill", "job_count"]
)


skill_gap_df = skill_gap_df.sort_values(
    by="job_count",
    ascending=False
).reset_index(drop=True)


print("✅ SKILL GAP ANALYSIS COMPLETED!")

display(skill_gap_df.head(20))

✅ SKILL GAP ANALYSIS COMPLETED!


,skill,job_count
0,none,1
1,blockchain,1
2,internet of things (iot),1
3,virtual reality (vr),1
4,augmented reality (ar),1
5,big data infrastructure,1
6,communication and presentation skills,1
7,cell counters,1
8,spent media analyzers,1
9,good documentation practices,1


In [30]:
# ============================================================
# CELL 30 — AI/ML SKILL-GAP FILTERING
# WHY:
# Remove irrelevant/domain-specific skills and keep only
# useful skills for an AI/ML career roadmap.
# ============================================================

AI_ML_SKILLS = {
    "python", "sql", "machine learning", "deep learning",
    "pytorch", "tensorflow", "scikit-learn", "nlp",
    "natural language processing", "computer vision",
    "transformers", "bert", "llm", "llms",
    "generative ai", "rag", "langchain", "langgraph",
    "embeddings", "vector database", "faiss", "chromadb",
    "data science", "statistics", "probability",
    "linear algebra", "model deployment", "mlops",
    "docker", "kubernetes", "fastapi", "api",
    "recommendation systems", "big data", "spark",
    "data engineering", "pandas", "numpy"
}

filtered_gaps = []

for skill in skill_gap_df["skill"]:

    skill_lower = skill.lower()

    if skill_lower == "none":
        continue

    if any(
        ai_skill in skill_lower or skill_lower in ai_skill
        for ai_skill in AI_ML_SKILLS
    ):
        filtered_gaps.append(skill)


filtered_skill_gap_df = pd.DataFrame(
    {
        "skill": filtered_gaps
    }
)

filtered_skill_gap_df["priority"] = range(
    len(filtered_skill_gap_df),
    0,
    -1
)

print("✅ AI/ML SKILL GAP FILTERING COMPLETED!")

display(filtered_skill_gap_df)

✅ AI/ML SKILL GAP FILTERING COMPLETED!


,skill,priority
0,big data infrastructure,1


In [31]:
# ============================================================
# CELL 31 — SEMANTIC SKILL GAP DETECTION
# WHY:
# Qwen compares every detected missing skill against the
# candidate's resume and decides whether it is genuinely
# missing, already covered, or irrelevant.
# ============================================================

candidate_skills_text = ", ".join(sorted(candidate_skills))

def validate_skill_gap(skill):

    prompt = f"""
You are an expert AI/ML career advisor.

Candidate skills:
{candidate_skills_text}

Potential missing skill:
{skill}

Determine whether this is a genuine skill gap for this
candidate's AI/ML career.

Return ONLY valid JSON:

{{
    "skill": "{skill}",
    "is_genuine_gap": true,
    "reason": "short explanation",
    "priority": "High"
}}

Rules:
- Do not mark a skill as missing if the candidate already
  has an equivalent or closely related skill.
- Ignore highly specialized skills unrelated to general
  AI/ML engineering.
- Priority must be High, Medium, or Low.
"""

    result = run_llm(prompt)
    return extract_json(result)


validated_gaps = []

for skill in skill_gap_df["skill"].head(20):

    if skill.lower() == "none":
        continue

    print(f"🔎 Checking: {skill}")

    result = validate_skill_gap(skill)

    if result:
        validated_gaps.append(result)


semantic_gap_df = pd.DataFrame(validated_gaps)

if not semantic_gap_df.empty:

    semantic_gap_df = semantic_gap_df[
        semantic_gap_df["is_genuine_gap"] == True
    ]

    priority_order = {
        "High": 3,
        "Medium": 2,
        "Low": 1
    }

    semantic_gap_df["priority_score"] = (
        semantic_gap_df["priority"]
        .map(priority_order)
    )

    semantic_gap_df = semantic_gap_df.sort_values(
        "priority_score",
        ascending=False
    ).reset_index(drop=True)


print("\n✅ SEMANTIC SKILL GAP ANALYSIS COMPLETED!")

display(semantic_gap_df)

🔎 Checking: blockchain
🔎 Checking: internet of things (iot)
🔎 Checking: virtual reality (vr)
🔎 Checking: augmented reality (ar)
🔎 Checking: big data infrastructure
🔎 Checking: communication and presentation skills
🔎 Checking: cell counters
🔎 Checking: spent media analyzers
🔎 Checking: good documentation practices
🔎 Checking: lab inventory management
🔎 Checking: kaizen and 5s/6s initiatives
🔎 Checking: transcriptomic data analysis
🔎 Checking: omics data integration
🔎 Checking: bioinformatics tools

✅ SEMANTIC SKILL GAP ANALYSIS COMPLETED!


,skill,is_genuine_gap,reason,priority,priority_score


In [32]:
# ============================================================
# CELL 32 — RELIABLE JD SKILL-GAP EXTRACTION
# WHY:
# Extract required skills directly from each top job and let
# Qwen compare them against the candidate's resume skills.
# ============================================================

candidate_profile = f"""
Resume:
{resume_text}

Detected candidate skills:
{candidate_skills_text}
"""

def extract_real_skill_gaps(row):

    prompt = f"""
You are an expert AI/ML recruitment analyst.

{candidate_profile}

JOB:
Title: {row['title']}
Description:
{row['job_text']}

Identify ONLY important technical skills explicitly required
or strongly implied by this job that the candidate does NOT
currently demonstrate in the resume.

Ignore:
- generic soft skills
- company-specific tools
- highly specialized domain skills unrelated to AI/ML
- skills already demonstrated through equivalent technologies

Return ONLY valid JSON:

{{
  "missing_skills": [
    {{
      "skill": "skill name",
      "importance": "High",
      "reason": "why it is missing"
    }}
  ]
}}
"""

    result = run_llm(prompt)
    return extract_json(result)


real_skill_gaps = []

for _, row in top_jobs.iterrows():

    print(f"🔍 Extracting gaps: {row['title']}")

    result = extract_real_skill_gaps(row)

    if result and isinstance(result.get("missing_skills"), list):

        for gap in result["missing_skills"]:

            gap["source_job"] = row["title"]
            real_skill_gaps.append(gap)


real_skill_gap_df = pd.DataFrame(real_skill_gaps)

print("\n✅ REAL AI/ML SKILL-GAP EXTRACTION COMPLETED!")

if not real_skill_gap_df.empty:
    display(real_skill_gap_df)
else:
    print("No genuine technical skill gaps detected.")

🔍 Extracting gaps: Research Internship
🔍 Extracting gaps: Emerging Technology Engineer
🔍 Extracting gaps: Data Engineer, Google Nest
🔍 Extracting gaps: Research Associate
🔍 Extracting gaps: Bioinformatician I/II

✅ REAL AI/ML SKILL-GAP EXTRACTION COMPLETED!


,skill,importance,reason,source_job
0,ACL Publications,High,"The candidate has not published in ACL, a key ...",Research Internship
1,CHI Publications,High,"The candidate has not published in CHI, a key ...",Research Internship
2,ICLR Publications,High,"The candidate has not published in ICLR, a key...",Research Internship
3,EMNLP Publications,High,"The candidate has not published in EMNLP, a ke...",Research Internship
4,SIGIR Publications,High,"The candidate has not published in SIGIR, a ke...",Research Internship
5,Java,High,The candidate has not listed Java as a program...,Emerging Technology Engineer
6,Blockchain,High,The candidate has not mentioned any experience...,Emerging Technology Engineer
7,IoT,High,The candidate has not listed any experience or...,Emerging Technology Engineer
8,VR,High,The candidate has not mentioned any experience...,Emerging Technology Engineer
9,AR,High,The candidate has not listed any experience or...,Emerging Technology Engineer


In [33]:
# ============================================================
# CELL 33 — CORE AI/ML SKILL GAP ANALYZER
# WHY:
# Remove job-specific/domain-specific requirements and identify
# only transferable AI/ML engineering skills useful for the
# candidate's target career.
# ============================================================

CORE_AI_ML_SKILLS = [
    "Python",
    "SQL",
    "Machine Learning",
    "Deep Learning",
    "PyTorch",
    "TensorFlow",
    "Scikit-learn",
    "NLP",
    "Computer Vision",
    "Transformers",
    "BERT",
    "LLM",
    "Generative AI",
    "RAG",
    "LangChain",
    "LangGraph",
    "Embeddings",
    "Vector Database",
    "FAISS",
    "ChromaDB",
    "FastAPI",
    "Streamlit",
    "Docker",
    "MLOps",
    "Model Deployment",
    "REST APIs",
    "Pandas",
    "NumPy",
    "Statistics",
    "Probability",
    "Linear Algebra",
    "Recommendation Systems",
    "Data Engineering",
    "Apache Spark",
    "AWS",
    "Azure",
    "GCP"
]

candidate_profile = f"""
Resume:
{resume_text}

Known candidate skills:
{candidate_skills_text}
"""

core_gap_results = []

for skill in CORE_AI_ML_SKILLS:

    prompt = f"""
You are evaluating an AI/ML fresher.

{candidate_profile}

Target skill:
{skill}

Determine whether the candidate's resume demonstrates
this skill.

Rules:
1. If clearly demonstrated -> status = "Already Have"
2. If partially demonstrated -> status = "Needs Improvement"
3. If not demonstrated -> status = "Missing"
4. Do not infer experience that is not present.
5. Consider equivalent technologies where appropriate.

Return ONLY JSON:

{{
    "skill": "{skill}",
    "status": "Already Have",
    "reason": "short evidence-based explanation"
}}
"""

    result = run_llm(prompt)

    if result:
        parsed = extract_json(result)

        if parsed:
            core_gap_results.append(parsed)


core_gap_df = pd.DataFrame(core_gap_results)

print("✅ CORE AI/ML SKILL ANALYSIS COMPLETED!")

display(core_gap_df)

✅ CORE AI/ML SKILL ANALYSIS COMPLETED!


,skill,status,reason
0,Python,Already Have,The candidate has listed Python as one of thei...
1,SQL,Missing,The candidate's resume does not mention any ex...
2,Machine Learning,Already Have,The candidate has extensive experience in vari...
3,Deep Learning,Already Have,The candidate has listed several deep learning...
4,PyTorch,Already Have,The candidate has listed PyTorch under their L...
5,TensorFlow,Missing,The candidate's resume does not mention Tensor...
6,Scikit-learn,Missing,The candidate's resume does not mention Scikit...
7,NLP,Already Have,"The candidate has extensive experience in NLP,..."
8,Computer Vision,Missing,The candidate's resume does not mention any ex...
9,Transformers,Already Have,The candidate has listed 'Transformers' under ...


In [34]:
# ============================================================
# CELL 34 — AI/ML SKILL PRIORITY RANKING
# WHY:
# Prioritize missing/improvable skills that have the highest
# value for an AI/ML Engineer career.
# ============================================================

HIGH_PRIORITY = {
    "SQL",
    "Scikit-learn",
    "Computer Vision",
    "LangChain",
    "LangGraph",
    "Vector Database",
    "FAISS",
    "Docker",
    "MLOps",
    "Model Deployment",
    "REST APIs",
    "Statistics",
    "Probability",
    "Linear Algebra"
}

MEDIUM_PRIORITY = {
    "TensorFlow",
    "ChromaDB",
    "Streamlit",
    "Recommendation Systems",
    "Data Engineering",
    "Apache Spark",
    "AWS",
    "Azure",
    "GCP"
}


def assign_priority(row):

    if row["status"] == "Already Have":
        return "Already Have"

    skill = row["skill"]

    if skill in HIGH_PRIORITY:
        return "High"

    if skill in MEDIUM_PRIORITY:
        return "Medium"

    return "Low"


core_gap_df["priority"] = core_gap_df.apply(
    assign_priority,
    axis=1
)

priority_order = {
    "High": 3,
    "Medium": 2,
    "Low": 1,
    "Already Have": 0
}

core_gap_df["priority_score"] = (
    core_gap_df["priority"].map(priority_order)
)

core_gap_df = core_gap_df.sort_values(
    "priority_score",
    ascending=False
).reset_index(drop=True)

print("✅ SKILL PRIORITY RANKING COMPLETED!")

display(
    core_gap_df[
        [
            "skill",
            "status",
            "priority",
            "reason"
        ]
    ]
)

✅ SKILL PRIORITY RANKING COMPLETED!


,skill,status,priority,reason
0,SQL,Missing,High,The candidate's resume does not mention any ex...
1,LangChain,Missing,High,The candidate has not listed LangChain as one ...
2,LangGraph,Missing,High,The candidate's resume does not mention any ex...
3,Computer Vision,Missing,High,The candidate's resume does not mention any ex...
4,Scikit-learn,Missing,High,The candidate's resume does not mention Scikit...
5,REST APIs,Missing,High,The candidate's resume does not mention any ex...
6,Linear Algebra,Missing,High,The candidate's resume does not mention any ex...
7,Probability,Missing,High,The candidate's resume does not mention any ex...
8,Statistics,Missing,High,The candidate's resume does not explicitly men...
9,Vector Database,Missing,High,The candidate's resume does not mention any ex...


In [36]:
# ============================================================
# CELL 35 — PERSONALIZED 30/60/90-DAY AI/ML ROADMAP
# WHY:
# Generate a practical roadmap from the prioritized skill
# gaps while considering the candidate's existing strengths.
# ============================================================

priority_gaps = core_gap_df[
    core_gap_df["priority"].isin(["High", "Medium"])
][["skill", "priority"]].to_dict("records")

existing_skills = core_gap_df[
    core_gap_df["status"] == "Already Have"
]["skill"].tolist()

roadmap_prompt = f"""
You are an expert AI/ML career mentor.

Candidate target:
AI/ML Engineer / Machine Learning Engineer

Existing skills:
{existing_skills}

Priority skill gaps:
{priority_gaps}

Create a practical 30/60/90-day learning roadmap.

Rules:
- Build on existing Python, PyTorch, NLP, LLM, RAG and ML knowledge.
- Prioritize skills that improve employability for an AI/ML Engineer.
- Do not repeat skills the candidate already knows.
- Include projects, not only theory.
- Include deployment and production skills.
- Keep the roadmap realistic for a fresher.
- Do not recommend unrelated domain-specific skills.

Return ONLY valid JSON:

{{
  "days_1_30": {{
    "focus": "...",
    "skills": [],
    "projects": [],
    "outcome": "..."
  }},
  "days_31_60": {{
    "focus": "...",
    "skills": [],
    "projects": [],
    "outcome": "..."
  }},
  "days_61_90": {{
    "focus": "...",
    "skills": [],
    "projects": [],
    "outcome": "..."
  }}
}}
"""

roadmap_result = run_llm(roadmap_prompt)

roadmap = extract_json(roadmap_result)

print("✅ PERSONALIZED 30/60/90-DAY ROADMAP GENERATED!")

roadmap

✅ PERSONALIZED 30/60/90-DAY ROADMAP GENERATED!


{'days_1_30': {'focus': 'Building foundational skills in SQL and basic MLOps to enhance data handling and model deployment capabilities.',
  'skills': ['SQL', 'Docker', 'Model Deployment'],
  'projects': ['Build a simple ETL pipeline using SQL to extract, transform, and load data from various sources into a database.',
   'Set up a Docker container for a machine learning model and deploy it using a simple Flask API.'],
  'outcome': 'Candidates will be able to handle data more effectively and deploy models in a containerized environment.'},
 'days_31_60': {'focus': 'Expanding knowledge in computer vision and advanced MLOps practices to improve model performance and scalability.',
  'skills': ['Computer Vision', 'FAISS', 'MLOps'],
  'projects': ['Implement a basic computer vision model using PyTorch and evaluate its performance on a dataset.',
   'Optimize a machine learning model using FAISS for efficient similarity search and implement it in a production environment.'],
  'outcome': 'C

In [37]:
# ============================================================
# CELL 36 — ROADMAP CORRECTION & VALIDATION
# WHY:
# Remove skills already implemented in the current project
# and align the roadmap with the actual AI Career Platform.
# ============================================================

implemented_skills = [
    "Python",
    "PyTorch",
    "NLP",
    "LLM",
    "Generative AI",
    "RAG",
    "Embeddings",
    "FAISS",
    "FastAPI",
    "LangChain",
    "LangGraph"
]

# Remove already implemented skills from roadmap
for phase in ["days_1_30", "days_31_60", "days_61_90"]:

    roadmap[phase]["skills"] = [
        skill
        for skill in roadmap[phase]["skills"]
        if skill not in implemented_skills
    ]

# Replace Flask with FastAPI where it appears
for phase in ["days_1_30", "days_31_60", "days_61_90"]:

    roadmap[phase]["projects"] = [
        project.replace("Flask API", "FastAPI")
        for project in roadmap[phase]["projects"]
    ]

print("✅ ROADMAP VALIDATED AND ALIGNED WITH PROJECT STACK!")

display(pd.DataFrame([
    {
        "Phase": "Days 1–30",
        "Focus": roadmap["days_1_30"]["focus"],
        "Skills": ", ".join(roadmap["days_1_30"]["skills"])
    },
    {
        "Phase": "Days 31–60",
        "Focus": roadmap["days_31_60"]["focus"],
        "Skills": ", ".join(roadmap["days_31_60"]["skills"])
    },
    {
        "Phase": "Days 61–90",
        "Focus": roadmap["days_61_90"]["focus"],
        "Skills": ", ".join(roadmap["days_61_90"]["skills"])
    }
]))

✅ ROADMAP VALIDATED AND ALIGNED WITH PROJECT STACK!


,Phase,Focus,Skills
0,Days 1–30,Building foundational skills in SQL and basic ...,"SQL, Docker, Model Deployment"
1,Days 31–60,Expanding knowledge in computer vision and adv...,"Computer Vision, MLOps"
2,Days 61–90,"Mastering advanced topics such as LangChain, L...","AWS, Recommendation Systems"


In [38]:
# ============================================================
# CELL 37 — BUILD CAREER RAG DOCUMENTS
# WHY:
# Create a knowledge base from jobs, skill analysis, and the
# personalized roadmap so the AI can retrieve relevant career
# information instead of relying only on the LLM's memory.
# ============================================================

from langchain_core.documents import Document

rag_documents = []

# ------------------------------------------------------------
# 1. Job documents
# ------------------------------------------------------------

for _, row in top_jobs.iterrows():

    rag_documents.append(
        Document(
            page_content=f"""
Job Title: {row['title']}
Location: {row['job_location']}
Work Type: {row['work_type']}
Role Type: {row['role_type']}

Job Description:
{row['job_text']}

Semantic Score: {row['semantic_score']:.2f}
Skill Match: {row['skill_match_score']:.2f}
Final Candidate Score: {row['final_candidate_score']:.2f}
""",
            metadata={
                "type": "job",
                "title": row["title"],
                "location": row["job_location"]
            }
        )
    )


# ------------------------------------------------------------
# 2. Skill-gap documents
# ------------------------------------------------------------

for _, row in core_gap_df.iterrows():

    rag_documents.append(
        Document(
            page_content=f"""
Skill: {row['skill']}
Status: {row['status']}
Priority: {row['priority']}

Reason:
{row['reason']}
""",
            metadata={
                "type": "skill_gap",
                "skill": row["skill"],
                "priority": row["priority"]
            }
        )
    )


# ------------------------------------------------------------
# 3. Roadmap documents
# ------------------------------------------------------------

for phase_key, phase_data in roadmap.items():

    phase_name = phase_key.replace("_", " ").title()

    rag_documents.append(
        Document(
            page_content=f"""
Career Roadmap Phase:
{phase_name}

Focus:
{phase_data['focus']}

Skills:
{", ".join(phase_data['skills'])}

Projects:
{" | ".join(phase_data['projects'])}

Expected Outcome:
{phase_data['outcome']}
""",
            metadata={
                "type": "roadmap",
                "phase": phase_name
            }
        )
    )


print("✅ RAG DOCUMENTS CREATED!")
print(f"Total documents: {len(rag_documents)}")

for doc in rag_documents[:3]:
    print("\n--- DOCUMENT ---")
    print(doc.page_content[:500])

✅ RAG DOCUMENTS CREATED!
Total documents: 45

--- DOCUMENT ---

Job Title: Research Internship
Location: United States
Work Type: Internship
Role Type: Research Internship

Job Description:
Job Title: Research Internship
Job Description: Remote Eligible: Persons in these roles are welcome to work remotely from any state in the US.
Compensation Range: $77,490 - $155,810*
Compensation dependent upon degree attained 
Research Internship
Who You Are:
As a Research Intern on our team, you’ll have the unique opportunity to see your research improve the productivi

--- DOCUMENT ---

Job Title: Emerging Technology Engineer
Location: Kalamazoo, MI
Work Type: Full-time
Role Type: Emerging Technology Engineer

Job Description:
Job Title: Emerging Technology Engineer
Job Description: Position Summary:Zoetis, the world leader in animal health, is seeking a highly skilled and motivated Emerging Technology Engineer to join our dynamic team. As an Emerging Technology Engineer, you will play a crucial 

In [39]:
# ============================================================
# CELL 38 — LANGCHAIN RAG VECTOR STORE + RETRIEVER
# WHY:
# Convert the career documents into embeddings and create a
# LangChain-compatible FAISS retriever for RAG.
# ============================================================

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Use the same BGE-large model already loaded
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

# Create LangChain FAISS vector store
rag_vectorstore = FAISS.from_documents(
    documents=rag_documents,
    embedding=embedding_model
)

# Create retriever
rag_retriever = rag_vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

print("✅ LANGCHAIN RAG VECTOR STORE CREATED!")
print(f"Documents indexed: {len(rag_documents)}")

# Test retrieval
test_query = "What skills should I learn to become an AI/ML Engineer?"

retrieved_docs = rag_retriever.invoke(test_query)

print("\n🔎 RETRIEVED DOCUMENTS:")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- RESULT {i} ---")
    print(doc.page_content[:400])

/tmp/ipykernel_2222/2121524296.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ LANGCHAIN RAG VECTOR STORE CREATED!
Documents indexed: 45

🔎 RETRIEVED DOCUMENTS:

--- RESULT 1 ---

Career Roadmap Phase:
Days 61 90

Focus:
Mastering advanced topics such as LangChain, LangGraph, and AWS to become proficient in modern AI/ML engineering practices.

Skills:
AWS, Recommendation Systems

Projects:
Develop a recommendation system using LangChain and evaluate its performance on a dataset. | Deploy a machine learning model on AWS using EC2 instances and set up monitoring and logging.

--- RESULT 2 ---

Career Roadmap Phase:
Days 31 60

Focus:
Expanding knowledge in computer vision and advanced MLOps practices to improve model performance and scalability.

Skills:
Computer Vision, MLOps

Projects:
Implement a basic computer vision model using PyTorch and evaluate its performance on a dataset. | Optimize a machine learning model using FAISS for efficient similarity search and implement it in a pr

--- RESULT 3 ---

Skill: Machine Learning
Status: Already Have
Priority: Alre

In [40]:
# ============================================================
# CELL 39 — RAG CAREER ADVISOR
# WHY:
# Combine LangChain retrieval with Qwen so answers are based
# on the candidate's actual jobs, skills, gaps and roadmap.
# ============================================================

def career_rag_answer(question):

    # Retrieve relevant career knowledge
    docs = rag_retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    prompt = f"""
You are an AI Career Advisor.

Answer the user's question using ONLY the retrieved
career context below.

RETRIEVED CONTEXT:
{context}

USER QUESTION:
{question}

Rules:
- Give practical and personalized advice.
- Do not invent information.
- If the context does not contain the answer, say so.
- Keep the answer concise.
"""

    answer = run_llm(prompt)

    return answer


# Test the RAG Career Advisor
question = "What should I learn first to become an AI/ML Engineer?"

answer = career_rag_answer(question)

print("🤖 RAG CAREER ADVISOR")
print("=" * 60)
print(answer)

🤖 RAG CAREER ADVISOR
{
  "advice": "Based on your current skillset, you should focus on expanding your knowledge in computer vision and advanced MLOps practices to improve model performance and scalability. This aligns with the skills and projects outlined for Days 31-60 in the career roadmap."
}


In [41]:
# ============================================================
# CELL 40 — LANGGRAPH CAREER ADVISOR WORKFLOW
# WHY:
# Use LangGraph to create a stateful AI workflow:
# Question → Retrieve → Analyze → Generate Answer
# ============================================================

from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class CareerState(TypedDict):
    question: str
    context: str
    answer: str


# ------------------------------------------------------------
# NODE 1 — Retrieve relevant career information
# ------------------------------------------------------------

def retrieve_career_context(state: CareerState):

    docs = rag_retriever.invoke(
        state["question"]
    )

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    return {
        "context": context
    }


# ------------------------------------------------------------
# NODE 2 — Generate AI career answer
# ------------------------------------------------------------

def generate_career_answer(state: CareerState):

    prompt = f"""
You are an AI Career Advisor.

Retrieved career information:
{state["context"]}

User question:
{state["question"]}

Answer using the retrieved information.
Be concise, practical and personalized.
Do not invent information.

Return ONLY JSON:

{{
    "answer": "your answer"
}}
"""

    result = run_llm(prompt)

    parsed = extract_json(result)

    if parsed:
        return {
            "answer": parsed.get("answer", result)
        }

    return {
        "answer": result
    }


# ------------------------------------------------------------
# BUILD LANGGRAPH
# ------------------------------------------------------------

career_graph = StateGraph(CareerState)

career_graph.add_node(
    "retrieve",
    retrieve_career_context
)

career_graph.add_node(
    "generate",
    generate_career_answer
)

career_graph.add_edge(
    START,
    "retrieve"
)

career_graph.add_edge(
    "retrieve",
    "generate"
)

career_graph.add_edge(
    "generate",
    END
)

career_app = career_graph.compile()

print("✅ LANGGRAPH CAREER WORKFLOW CREATED!")


# ------------------------------------------------------------
# TEST WORKFLOW
# ------------------------------------------------------------

test_question = "What should I learn in the next 30 days?"

result = career_app.invoke({
    "question": test_question
})

print("\n🤖 LANGGRAPH ANSWER")
print("=" * 60)
print(result["answer"])

✅ LANGGRAPH CAREER WORKFLOW CREATED!

🤖 LANGGRAPH ANSWER
Build a simple ETL pipeline using SQL to extract, transform, and load data from various sources into a database. Set up a Docker container for a machine learning model and deploy it using a simple FastAPI.


In [42]:
# ============================================================
# CELL 41 — LANGGRAPH ADAPTIVE MOCK INTERVIEW
# WHY:
# Create a stateful interview engine that asks questions,
# evaluates answers, and automatically adjusts difficulty.
# ============================================================

class InterviewState(TypedDict):
    candidate_skills: str
    question: str
    answer: str
    score: int
    feedback: str
    difficulty: str
    next_action: str


# ------------------------------------------------------------
# NODE 1 — Generate interview question
# ------------------------------------------------------------

def generate_interview_question(state: InterviewState):

    prompt = f"""
You are a technical AI/ML interviewer.

Candidate skills:
{state["candidate_skills"]}

Difficulty:
{state["difficulty"]}

Generate ONE technical interview question for an
AI/ML Engineer candidate.

Return ONLY JSON:

{{
    "question": "interview question"
}}
"""

    result = run_llm(prompt)
    parsed = extract_json(result)

    return {
        "question": parsed["question"]
        if parsed else result
    }


# ------------------------------------------------------------
# NODE 2 — Evaluate candidate answer
# ------------------------------------------------------------

def evaluate_interview_answer(state: InterviewState):

    prompt = f"""
You are an expert AI/ML interviewer.

Question:
{state["question"]}

Candidate Answer:
{state["answer"]}

Evaluate the answer.

Return ONLY JSON:

{{
    "score": 0,
    "feedback": "specific technical feedback",
    "next_action": "increase / maintain / decrease"
}}

Score from 0 to 100.

Rules:
- 80-100 → increase difficulty
- 50-79 → maintain difficulty
- 0-49 → decrease difficulty
"""

    result = run_llm(prompt)
    parsed = extract_json(result)

    if parsed:
        return {
            "score": int(parsed.get("score", 0)),
            "feedback": parsed.get("feedback", ""),
            "next_action": parsed.get(
                "next_action",
                "maintain"
            )
        }

    return {
        "score": 0,
        "feedback": result,
        "next_action": "maintain"
    }


# ------------------------------------------------------------
# NODE 3 — Adapt difficulty
# ------------------------------------------------------------

def adapt_interview_difficulty(state: InterviewState):

    action = state["next_action"]

    if action == "increase":
        difficulty = "Hard"

    elif action == "decrease":
        difficulty = "Easy"

    else:
        difficulty = state["difficulty"]

    return {
        "difficulty": difficulty
    }


# ------------------------------------------------------------
# BUILD LANGGRAPH
# ------------------------------------------------------------

interview_graph = StateGraph(InterviewState)

interview_graph.add_node(
    "generate_question",
    generate_interview_question
)

interview_graph.add_node(
    "evaluate_answer",
    evaluate_interview_answer
)

interview_graph.add_node(
    "adapt_difficulty",
    adapt_interview_difficulty
)

interview_graph.add_edge(
    START,
    "generate_question"
)

interview_graph.add_edge(
    "generate_question",
    "evaluate_answer"
)

interview_graph.add_edge(
    "evaluate_answer",
    "adapt_difficulty"
)

interview_graph.add_edge(
    "adapt_difficulty",
    END
)

interview_app = interview_graph.compile()

print("✅ ADAPTIVE MOCK INTERVIEW LANGGRAPH CREATED!")

✅ ADAPTIVE MOCK INTERVIEW LANGGRAPH CREATED!


In [43]:
# ============================================================
# CELL 42 — TEST ADAPTIVE MOCK INTERVIEW
# WHY:
# Run one complete interview round:
# Generate Question → Submit Answer → Evaluate → Adapt
# ============================================================

candidate_skills = ", ".join(
    core_gap_df[
        core_gap_df["status"] == "Already Have"
    ]["skill"].tolist()
)

# Generate the first question
question_result = interview_app.invoke({
    "candidate_skills": candidate_skills,
    "question": "",
    "answer": "",
    "score": 0,
    "feedback": "",
    "difficulty": "Medium",
    "next_action": "maintain"
})

question = question_result["question"]

print("🎤 INTERVIEW QUESTION")
print("=" * 60)
print(question)


# ------------------------------------------------------------
# Enter candidate answer
# ------------------------------------------------------------

candidate_answer = input(
    "\n✍️ Enter your answer: "
)


# ------------------------------------------------------------
# Evaluate answer
# ------------------------------------------------------------

evaluation_result = interview_app.invoke({
    "candidate_skills": candidate_skills,
    "question": question,
    "answer": candidate_answer,
    "score": 0,
    "feedback": "",
    "difficulty": "Medium",
    "next_action": "maintain"
})


print("\n📊 INTERVIEW RESULT")
print("=" * 60)

print("Score:", evaluation_result["score"])
print("Feedback:", evaluation_result["feedback"])
print("Next Action:", evaluation_result["next_action"])
print("Next Difficulty:", evaluation_result["difficulty"])

🎤 INTERVIEW QUESTION
Explain the process of fine-tuning a pre-trained Transformer model like BERT for a custom NLP task using PyTorch and FastAPI. Include how you would handle data preprocessing, model training, and integrating the model into a web service.

✍️ Enter your answer: I would first preprocess the custom NLP dataset by cleaning the text, creating train/validation/test splits, and using BERT’s tokenizer to convert text into input_ids, attention_mask, and labels.   Using PyTorch and Hugging Face Transformers, I would load a pre-trained BERT model such as BertForSequenceClassification, replace the classification head according to the number of classes, and fine-tune it using AdamW, a small learning rate, mini-batches, validation, and early stopping. I would evaluate it using accuracy, precision, recall, and F1-score.

📊 INTERVIEW RESULT
Score: 79
Feedback: The candidate provided a generally accurate overview of the fine-tuning process for a BERT model using PyTorch and Hugging 

In [44]:
# ============================================================
# CELL 43 — NEXT ADAPTIVE INTERVIEW ROUND
# WHY:
# Continue the interview using the previous score to decide
# whether the next question should become easier, stay the
# same, or become harder.
# ============================================================

previous_score = evaluation_result["score"]
previous_difficulty = evaluation_result["difficulty"]

# Generate next question based on previous performance
next_round = interview_app.invoke({
    "candidate_skills": candidate_skills,
    "question": "",
    "answer": "",
    "score": previous_score,
    "feedback": evaluation_result["feedback"],
    "difficulty": previous_difficulty,
    "next_action": evaluation_result["next_action"]
})

next_question = next_round["question"]

print("🎤 NEXT ADAPTIVE INTERVIEW QUESTION")
print("=" * 60)
print(next_question)

candidate_answer_2 = input(
    "\n✍️ Enter your answer: "
)

# Evaluate the new answer
second_result = interview_app.invoke({
    "candidate_skills": candidate_skills,
    "question": next_question,
    "answer": candidate_answer_2,
    "score": 0,
    "feedback": "",
    "difficulty": previous_difficulty,
    "next_action": "maintain"
})

print("\n📊 ROUND 2 RESULT")
print("=" * 60)
print("Score:", second_result["score"])
print("Feedback:", second_result["feedback"])
print("Next Action:", second_result["next_action"])
print("Next Difficulty:", second_result["difficulty"])

🎤 NEXT ADAPTIVE INTERVIEW QUESTION
Explain the process of fine-tuning a pre-trained Transformer model like BERT for a custom NLP task using PyTorch and FastAPI. Include how you would handle data preprocessing, model training, and integrating the model into a web service.

✍️ Enter your answer: I would first preprocess the custom NLP dataset by cleaning the text, creating train/validation/test splits, and using BERT’s tokenizer to convert text into input_ids, attention_mask, and labels.   Using PyTorch and Hugging Face Transformers, I would load a pre-trained BERT model such as BertForSequenceClassification, replace the classification head according to the number of classes, and fine-tune it using AdamW, a small learning rate, mini-batches, validation, and early stopping. I would evaluate it using accuracy, precision, recall, and F1-score.   After training, I would save the tokenizer and model. For deployment, I would create a FastAPI endpoint that accepts text, tokenizes it using the s

In [45]:
# ============================================================
# CELL 44 — HARD AI/ML INTERVIEW ROUND
# WHY:
# Test advanced reasoning and system-design ability after
# the candidate successfully scored 85 in the previous round.
# ============================================================

hard_question_result = interview_app.invoke({
    "candidate_skills": candidate_skills,
    "question": "",
    "answer": "",
    "score": second_result["score"],
    "feedback": second_result["feedback"],
    "difficulty": "Hard",
    "next_action": "increase"
})

hard_question = hard_question_result["question"]

print("🔥 HARD AI/ML INTERVIEW QUESTION")
print("=" * 60)
print(hard_question)

hard_answer = input(
    "\n✍️ Enter your answer: "
)

hard_result = interview_app.invoke({
    "candidate_skills": candidate_skills,
    "question": hard_question,
    "answer": hard_answer,
    "score": 0,
    "feedback": "",
    "difficulty": "Hard",
    "next_action": "maintain"
})

print("\n📊 HARD ROUND RESULT")
print("=" * 60)
print("Score:", hard_result["score"])
print("Feedback:", hard_result["feedback"])
print("Next Action:", hard_result["next_action"])
print("Next Difficulty:", hard_result["difficulty"])

🔥 HARD AI/ML INTERVIEW QUESTION
Explain the concept of transfer learning in machine learning and provide an example of how you would implement it using PyTorch to fine-tune a pre-trained BERT model on a specific NLP task such as sentiment analysis. Discuss any challenges you encountered and how you addressed them.

✍️ Enter your answer: Transfer learning means using knowledge learned by a model on a large dataset and adapting it to a specific downstream task. For sentiment analysis, I would use a pre-trained BERT model and fine-tune it on a labeled sentiment dataset such as positive/negative reviews. First, I would clean the text, split it into train/validation/test sets, and use BERT’s tokenizer to generate input_ids and attention_mask. In PyTorch, I would load BertForSequenceClassification with two output classes and fine-tune it using AdamW, a small learning rate, mini-batches, validation, and early stopping. I would evaluate the model using accuracy, precision, recall, and F1-score

In [46]:
# ============================================================
# CELL 45 — ADVANCED IMPLEMENTATION INTERVIEW ROUND
# WHY:
# Test whether the candidate can explain actual PyTorch
# implementation details rather than only theory.
# ============================================================

implementation_prompt = f"""
You are a senior AI/ML interviewer.

Candidate skills:
{candidate_skills}

Previous performance:
Score: {hard_result["score"]}
Feedback: {hard_result["feedback"]}

Generate ONE difficult implementation-focused question.

The question must require the candidate to explain:
- PyTorch implementation
- model architecture or training pipeline
- data handling
- evaluation
- one practical production challenge

Avoid repeating the previous BERT transfer-learning question.

Return ONLY JSON:

{{
    "question": "advanced implementation interview question"
}}
"""

result = run_llm(implementation_prompt)

parsed = extract_json(result)

advanced_question = (
    parsed["question"]
    if parsed
    else result
)

print("🚀 ADVANCED IMPLEMENTATION QUESTION")
print("=" * 60)
print(advanced_question)

🚀 ADVANCED IMPLEMENTATION QUESTION
Advanced Implementation Interview Question: Implement a custom generative model using PyTorch for text generation from a given dataset. Your task is to design and implement the model architecture, including the choice of layers, activation functions, and loss function. Additionally, provide a detailed explanation of how you preprocess the text data, handle the training process, and evaluate the model's performance. Finally, discuss a practical challenge that might arise when deploying this model in a production environment and propose a solution.


In [47]:
# ============================================================
# CELL 46 — FINAL INTERVIEW REPORT
# WHY:
# Summarize interview performance across all completed rounds
# and identify strengths, weaknesses, and improvement areas.
# ============================================================

interview_scores = [
    evaluation_result["score"],
    second_result["score"],
    hard_result["score"]
]

average_score = sum(interview_scores) / len(interview_scores)

report_prompt = f"""
You are an expert AI/ML interview evaluator.

Candidate:
{candidate_skills}

Interview scores:
{interview_scores}

Average score:
{average_score:.2f}

Previous feedback:
1. {evaluation_result["feedback"]}
2. {second_result["feedback"]}
3. {hard_result["feedback"]}

Generate a concise final interview report.

Return ONLY JSON:

{{
    "overall_score": 0,
    "level": "Junior AI/ML Engineer",
    "strengths": [],
    "weaknesses": [],
    "technical_gaps": [],
    "improvement_plan": [],
    "interview_readiness": "Ready / Almost Ready / Needs Improvement"
}}
"""

report_result = run_llm(report_prompt)

interview_report = extract_json(report_result)

print("✅ FINAL INTERVIEW REPORT GENERATED!")
print("=" * 60)

display(pd.DataFrame({
    "Metric": [
        "Round 1",
        "Round 2",
        "Hard Round",
        "Average"
    ],
    "Score": [
        evaluation_result["score"],
        second_result["score"],
        hard_result["score"],
        round(average_score, 2)
    ]
}))

print("\n🤖 AI INTERVIEW REPORT")
print(interview_report)

✅ FINAL INTERVIEW REPORT GENERATED!


,Metric,Score
0,Round 1,79.0
1,Round 2,85.0
2,Hard Round,85.0
3,Average,83.0



🤖 AI INTERVIEW REPORT
{'overall_score': 83.0, 'level': 'Junior AI/ML Engineer', 'strengths': ['Demonstrated strong knowledge in machine learning, deep learning, and natural language processing.', 'Familiarity with popular libraries such as PyTorch, FastAPI, and Hugging Face Transformers.', 'Ability to explain complex concepts like fine-tuning, transfer learning, and model deployment.'], 'weaknesses': ['Could provide more detailed and specific information on handling imbalanced datasets and hyperparameter tuning.', 'Lack of specific code snippets or implementation details in some areas.'], 'technical_gaps': ['More detailed explanation on handling imbalanced datasets and the use of callbacks in PyTorch.', 'Specific examples of hyperparameter tuning and cross-validation in practice.', 'More in-depth coverage of the exact implementation steps and code snippets for model deployment with FastAPI.'], 'improvement_plan': ['Review and practice handling imbalanced datasets and hyperparameter tu

In [48]:
# ============================================================
# CELL 47 — FASTAPI BACKEND
# WHY:
# Expose the AI Career Assistant through REST APIs so the
# Streamlit application running in VS Code can communicate
# with the Colab AI backend.
# ============================================================

from fastapi import FastAPI
from pydantic import BaseModel
from typing import Optional
import nest_asyncio
import uvicorn

nest_asyncio.apply()

api = FastAPI(
    title="AI Career Assistant API",
    description="AI-powered career analysis and recommendation API",
    version="1.0.0"
)


# ------------------------------------------------------------
# REQUEST MODELS
# ------------------------------------------------------------

class CareerQuestion(BaseModel):
    question: str


class InterviewRequest(BaseModel):
    question: str
    answer: str
    difficulty: str = "Medium"


# ------------------------------------------------------------
# HEALTH CHECK
# ------------------------------------------------------------

@api.get("/")
def home():

    return {
        "status": "online",
        "service": "AI Career Assistant",
        "version": "1.0.0"
    }


# ------------------------------------------------------------
# JOB RECOMMENDATIONS
# ------------------------------------------------------------

@api.get("/jobs")
def get_jobs():

    results = fresher_jobs.head(10).copy()

    return results[
        [
            "title",
            "job_location",
            "work_type",
            "final_candidate_score"
        ]
    ].to_dict(orient="records")


# ------------------------------------------------------------
# SKILL ANALYSIS
# ------------------------------------------------------------

@api.get("/skills")
def get_skills():

    results = core_gap_df[
        [
            "skill",
            "status",
            "priority"
        ]
    ]

    return results.to_dict(orient="records")


# ------------------------------------------------------------
# CAREER ROADMAP
# ------------------------------------------------------------

@api.get("/roadmap")
def get_roadmap():

    return roadmap


# ------------------------------------------------------------
# RAG CAREER ADVISOR
# ------------------------------------------------------------

@api.post("/career-advisor")
def career_advisor(request: CareerQuestion):

    answer = career_rag_answer(
        request.question
    )

    return {
        "question": request.question,
        "answer": answer
    }


# ------------------------------------------------------------
# INTERVIEW EVALUATION
# ------------------------------------------------------------

@api.post("/interview/evaluate")
def evaluate_interview(request: InterviewRequest):

    result = interview_app.invoke({

        "candidate_skills": candidate_skills,

        "question": request.question,

        "answer": request.answer,

        "score": 0,

        "feedback": "",

        "difficulty": request.difficulty,

        "next_action": "maintain"
    })

    return {
        "score": result["score"],
        "feedback": result["feedback"],
        "next_action": result["next_action"],
        "next_difficulty": result["difficulty"]
    }


print("✅ FASTAPI APPLICATION CREATED!")
print("Endpoints:")
print("GET  /")
print("GET  /jobs")
print("GET  /skills")
print("GET  /roadmap")
print("POST /career-advisor")
print("POST /interview/evaluate")

✅ FASTAPI APPLICATION CREATED!
Endpoints:
GET  /
GET  /jobs
GET  /skills
GET  /roadmap
POST /career-advisor
POST /interview/evaluate


In [50]:
# ============================================================
# CELL 48 — START FASTAPI + NGROK
# WHY:
# Create a public HTTPS URL for the Colab FastAPI backend so
# the Streamlit application running in VS Code can access it.
# ============================================================

from pyngrok import ngrok
import threading
import time

# ------------------------------------------------------------
# Add your ngrok authentication token
# ------------------------------------------------------------

NGROK_AUTH_TOKEN = "3FGEkZXZMJeYDS6Wbrb9ElUndk5_NubJDLgoVffQYWpimk2M"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)


# ------------------------------------------------------------
# Start FastAPI in background
# ------------------------------------------------------------

def start_api():

    uvicorn.run(
        api,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )


api_thread = threading.Thread(
    target=start_api,
    daemon=True
)

api_thread.start()

time.sleep(3)


# ------------------------------------------------------------
# Create public tunnel
# ------------------------------------------------------------

public_url = ngrok.connect(
    8000,
    "http"
)

print("🚀 FASTAPI SERVER RUNNING!")
print("=" * 60)
print("Public API URL:")
print(public_url)
print("\nSwagger documentation:")
print(f"{public_url}/docs")

INFO:     Started server process [2222]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


🚀 FASTAPI SERVER RUNNING!
Public API URL:
NgrokTunnel: "https://asleep-cohesive-payment.ngrok-free.dev" -> "http://localhost:8000"

Swagger documentation:
NgrokTunnel: "https://asleep-cohesive-payment.ngrok-free.dev" -> "http://localhost:8000"/docs


In [51]:
# ============================================================
# CELL 49 — VERIFY EXISTING FASTAPI + NGROK SERVER
# WHY:
# The FastAPI server is already running on port 8000.
# We only need to test the existing public endpoint.
# ============================================================

import requests

BASE_URL = "https://asleep-cohesive-payment.ngrok-free.dev"

response = requests.get(f"{BASE_URL}/")

print("Status:", response.status_code)
print("Response:", response.json())

print("\n🚀 API URL:")
print(BASE_URL)

print("\n📚 Swagger:")
print(f"{BASE_URL}/docs")

INFO:     136.85.116.116:0 - "GET / HTTP/1.1" 200 OK
Status: 200
Response: {'status': 'online', 'service': 'AI Career Assistant', 'version': '1.0.0'}

🚀 API URL:
https://asleep-cohesive-payment.ngrok-free.dev

📚 Swagger:
https://asleep-cohesive-payment.ngrok-free.dev/docs


In [ ]:
# ============================================================
# CELL 51 — RESUME UPLOAD API
# WHY:
# Accept a PDF resume from Streamlit and extract its text
# through the FastAPI backend.
# ============================================================

from fastapi import UploadFile, File, HTTPException
import fitz

@api.post("/upload-resume")
async def upload_resume(file: UploadFile = File(...)):

    if not file.filename.lower().endswith(".pdf"):
        raise HTTPException(
            status_code=400,
            detail="Only PDF resumes are supported."
        )

    file_bytes = await file.read()

    try:
        pdf = fitz.open(
            stream=file_bytes,
            filetype="pdf"
        )

        extracted_text = ""

        for page in pdf:
            extracted_text += page.get_text()

        pdf.close()

        if not extracted_text.strip():
            raise HTTPException(
                status_code=400,
                detail="Could not extract text from resume."
            )

        return {
            "filename": file.filename,
            "message": "Resume uploaded successfully",
            "characters": len(extracted_text),
            "resume_text": extracted_text
        }

    except Exception as e:

        raise HTTPException(
            status_code=500,
            detail=str(e)
        )


print("✅ RESUME UPLOAD ENDPOINT ADDED!")
print("POST /upload-resume")